In [1]:
%load_ext autoreload
%autoreload 2
%aimport
%matplotlib inline

Modules to reload:
all-except-skipped

Modules to skip:



In [2]:
import os
import sys
nb_dir = os.path.dirname(os.path.split(os.getcwd())[0])
if nb_dir not in sys.path:
    sys.path.append(nb_dir)

In [3]:
from tqdm import tqdm_notebook as tqdm
import pandas as pd
from turicreate import SFrame, load_sframe
from pathlib import Path
import turicreate.aggregate as agg


In [4]:
import numpy as np
from ScienceDynamics.config import EXTENDED_PAPERS_SFRAME

from ScienceDynamics.utils import detect_lang, filter_sframe_by_func
import numpy as np
import matplotlib.pyplot as plt


## Disease List

In [25]:
diseases_list = SFrame.read_csv("Data/diseases_list.csv")

Finished parsing file /storage/homedir/dima/Projects/ScienceDynamics/examples/Infectious Disease Trends/Data/diseases_list.csv

Parsing completed. Parsed 100 lines in 0.055029 secs.

------------------------------------------------------
Inferred types from first 100 line(s) of file as 
column_type_hints=[str,str,str,str,str,str,str]
If parsing fails due to incorrect types, you can correct
the inferred type list above and pass it to read_csv in
the column_type_hints argument
------------------------------------------------------


Finished parsing file /storage/homedir/dima/Projects/ScienceDynamics/examples/Infectious Disease Trends/Data/diseases_list.csv

Parsing completed. Parsed 334 lines in 0.022853 secs.

In [26]:
def get_diseases_by_datasource(diseases_list, source="GDB"):
    diseases = diseases_list[[source]].unique()
    diseases = diseases[diseases[source]!=""]
    diseases["id"] = range(len(diseases))
    return diseases_list.join(diseases)

In [27]:
def normalize_diseases_list(sf):
    sf = sf.pack_columns(["GIDEON","modified GIDEON","Alias","Alias2","Alias3"], new_column_name="diseases").stack('diseases', new_column_name='diseases')
    sf = sf[sf["diseases"]!=""]
    sf["diseases"] = sf["diseases"].apply(lambda x: x.lower())
    return sf[["id", "diseases"]].unique()

In [28]:
gideon_diseases = get_diseases_by_datasource(diseases_list, source="GIDEON")
diseases_id = normalize_diseases_list(gideon_diseases)

In [29]:
gideon_diseases[["id","GIDEON"]].rename({"GIDEON":"disease"}).save("Data/disease_names.csv")

In [57]:
gideon_diseases[gideon_diseases["GDB"]!=""].rename({"GIDEON":"disease"})[["id","disease","GDB"]].save("Data/disease_gdb.csv")

In [ ]:
gdb_diseases[["id","GIDEON"]].rename({"GIDEON":"disease"}).save("Data/disease_names.csv")

In [30]:
disease_set = set(diseases_id["diseases"])

In [31]:
diseases_id.save("Data/diseases_id.csv","csv")

In [45]:
diseases_id = SFrame.read_csv("Data/diseases_id.csv")
disease_set = set(diseases_id["diseases"])

Finished parsing file /storage/homedir/dima/Projects/ScienceDynamics/examples/Infectious Disease Trends/Data/diseases_id.csv

------------------------------------------------------
Inferred types from first 100 line(s) of file as 
column_type_hints=[str,int]
If parsing fails due to incorrect types, you can correct
the inferred type list above and pass it to read_csv in
the column_type_hints argument
------------------------------------------------------

Parsing completed. Parsed 100 lines in 0.028876 secs.

Finished parsing file /storage/homedir/dima/Projects/ScienceDynamics/examples/Infectious Disease Trends/Data/diseases_id.csv

Parsing completed. Parsed 582 lines in 0.022729 secs.

## Real-World

### ProMED Data Preperation

In [42]:
if Path("/storage/homedir/dima/Projects/Promed/data/promed_data.csv").exists():
    promed = SFrame.read_csv("/storage/homedir/dima/Projects/Promed/data/promed_data.csv")
else:
    promed = pd.concat([pd.read_csv(f) for f in Path("/storage/homedir/dima/Projects/Promed/data/csv/").glob("*.csv")])

Finished parsing file /storage/homedir/dima/Projects/Promed/data/promed_data.csv

Parsing completed. Parsed 100 lines in 0.488832 secs.

------------------------------------------------------
Inferred types from first 100 line(s) of file as 
column_type_hints=[str,str,str,str,str,int,str,str,str,str,str,str,int,str,float,str,float,str,str,str,str,str,float,float,str,str,str,str]
If parsing fails due to incorrect types, you can correct
the inferred type list above and pass it to read_csv in
the column_type_hints argument
------------------------------------------------------


Finished parsing file /storage/homedir/dima/Projects/Promed/data/promed_data.csv

Parsing completed. Parsed 59725 lines in 0.558459 secs.

In [43]:
def clean_promed(promed):
    promed = promed.copy()
#     promed = promed[promed["summary"].apply(lambda x: "update" not in x.lower())]
    promed["issue_date"] = promed["issue_date"].str_to_datetime()
    promed = promed.split_datetime("issue_date" ,column_name_prefix="")
    return promed[["alert_id", "disease_name", "year"]].unique()

In [46]:
promed = clean_promed(promed)

In [47]:
promed["disease_name"] = promed["disease_name"].apply(lambda x: x.lower())

In [48]:
promed = promed.join(diseases_id, on={"disease_name":"diseases"})

In [52]:
def generate_diseases_sf(promed, min_count=100):
    diseases = promed.groupby(["id","year"], {"promed_count": agg.COUNT()})
#     diseases= diseases[diseases["year"]<2015]
    diseases_sum = diseases.groupby("id", {"sum": agg.SUM("promed_count")})
    diseases_sum = diseases_sum.sort('sum', False)
    diseases_sum = diseases_sum[diseases_sum['sum']>min_count]
    diseases_sum = diseases_sum.join(diseases).remove_column("sum")
    return diseases_sum

In [53]:
promed_diseases_agg =  generate_diseases_sf(promed)

In [54]:
promed_diseases_agg

id,year,promed_count
3,2010,9
18,2012,4
51,2009,38
37,2018,11
25,2012,20
10,2019,47
51,2006,8
24,2014,42
1,2011,24
9,2010,55


In [55]:
promed_diseases_agg.save("Data/promed/promed_diseases_yearly.csv","csv")

### GBD

In [8]:
gbd = SFrame.read_csv("Data/gbd/gbd.csv")

Finished parsing file /storage/homedir/dima/Projects/ScienceDynamics/examples/Infectious Disease Trends/Data/gbd/gbd.csv

Parsing completed. Parsed 100 lines in 0.050604 secs.

------------------------------------------------------
Inferred types from first 100 line(s) of file as 
column_type_hints=[str,str,str,str,str,str,int,float,float,float]
If parsing fails due to incorrect types, you can correct
the inferred type list above and pass it to read_csv in
the column_type_hints argument
------------------------------------------------------


Finished parsing file /storage/homedir/dima/Projects/ScienceDynamics/examples/Infectious Disease Trends/Data/gbd/gbd.csv

Parsing completed. Parsed 7056 lines in 0.034351 secs.

## Bibliometric Datasets

In [14]:
import re

def ngrams(tokens, n):
    # Use the zip function to help us generate n-grams
    # Concatentate the tokens into ngrams and return
    ngrams = zip(*[tokens[i:] for i in range(n)])
    return [" ".join(ngram) for ngram in ngrams]


def word_ngrams(s, max_len=None,  min_len=0):
    s = s.lower()
    s = re.sub(r'[^a-zA-Z0-9\s]', ' ', s)
    tokens = [token for token in s.split(" ") if token != ""]
    if max_len is None:
        max_len = len(tokens)
    elif max_len > len(tokens):
        max_len = len(tokens)
    return {t for i in range(min_len, max_len) for t in ngrams(tokens, i+1)}

In [15]:
def fiter_diseases(diseases_sf, disease_set, max_len):
    diseases_sf["diseases"] = diseases_sf["Normalized paper title"].apply(lambda t: disease_set & word_ngrams(t, max_len) )
    diseases_sf = diseases_sf[diseases_sf["diseases"]!=[]]
    return diseases_sf.stack("diseases",new_column_name="disease")

In [16]:
max_len = max([len(d.split(" ")) for d in disease_set])

### Microsoft Academics

In [12]:
from ScienceDynamics.datasets.microsoft_academic_graph import MicrosoftAcademicGraph
from ScienceDynamics.config.configs import DATASETS_BASE_DIR
mag = MicrosoftAcademicGraph(DATASETS_BASE_DIR / "MicrosoftAcademicGraph.zip")

In [13]:
med_papers = mag.extended_papers
med_papers = med_papers[med_papers["Fields of study parent list names (L0)"].apply(lambda x: 1 if {'Medicine','Biology'} & set(x) else 0)]

In [17]:
diseases_mag = fiter_diseases(med_papers, disease_set, max_len)

In [18]:
diseases_mag = diseases_mag.join(diseases_id, on={"disease":"diseases"})

In [19]:
diseases_mag.save("Data/mag/diseases_mag.sframe")

## PubMed

In [24]:
from pathlib import Path
if not Path("Data/pubmed/diseases_pubmed.sframe").exists():
    pubmed = SFrame.read_json("Data/pubmed/pubmed.json")
    pubmed = pubmed[pubmed["pubdate"]!=""]
    pubmed["Normalized paper title"] = pubmed["title"].apply(lambda x: x.lower().replace("[","").replace("].",""))
    pubmed["pubdate"] = pubmed["pubdate"].astype(int)
    pubmed = pubmed.rename({"pubdate":"year"})
    pubmed.save("Data/pubmed/pubmed.sframe")
    diseases_pubmed = fiter_diseases(pubmed,  disease_set, max_len)
    diseases_pubmed = diseases_pubmed.join(diseases_id)
    diseases_pubmed.save("Data/pubmed/diseases_pubmed.sframe")
else:
    diseases_pubmed = load_sframe("Data/pubmed/diseases_pubmed.sframe")

In [248]:
pubmed.save("Data/pubmed/pubmed.sframe")

In [12]:
pubmed = load_sframe("Data/pubmed/pubmed.sframe")


In [13]:
pubmed["pmid"] = pubmed["pmid"].astype(int)

In [14]:
pubmed[pubmed["pmid"]==19597009]

Columns:
	abstract	str
	affiliations	str
	authors	str
	chemical_list	str
	country	str
	delete	str
	doi	str
	issn_linking	str
	journal	str
	keywords	str
	medline_ta	str
	mesh_terms	str
	nlm_unique_id	str
	other_id	str
	pmc	str
	pmid	int
	year	int
	publication_types	str
	title	str
	Normalized paper title	str

Rows: Unknown

Data:
+-------------------------------+-------------------------------+
|            abstract           |          affiliations         |
+-------------------------------+-------------------------------+
| An estimated 750 million p... | Department of Medical Para... |
+-------------------------------+-------------------------------+
+---------------------+---------------+---------------+--------+
|       authors       | chemical_list |    country    | delete |
+---------------------+---------------+---------------+--------+
| J Keiser;J Utzinger |               | United States | false  |
+---------------------+---------------+---------------+--------+
+----------------------+--------------+-------------------------------+----------+
|         doi          | issn_linking |            journal            | keywords |
+----------------------+--------------+-------------------------------+----------+
| 10.1128/CMR.00012-09 |  0893-8512   | Clinical microbiology reviews |          |
+----------------------+--------------+-------------------------------+----------+
+--------------------+-------------------------------+---------------+----------+
|     medline_ta     |           mesh_terms          | nlm_unique_id | other_id |
+--------------------+-------------------------------+---------------+----------+
| Clin Microbiol Rev | D000818:Animals; D016266:F... |    8807282    |          |
+--------------------+-------------------------------+---------------+----------+
+-----+----------+------+-------------------------------+
| pmc |   pmid   | year |       publication_types       |
+-----+----------+------+-------------------------------+
|     | 19597009 | 2009 | D016428:Journal Article; D... |
+-----+----------+------+-------------------------------+
+---------------------------+---------------------------+
|           title           |   Normalized paper title  |
+---------------------------+---------------------------+
| Food-borne trematodiases. | food-borne trematodiases. |
+---------------------------+---------------------------+
[? rows x 20 columns]
Note: Only the head of the SFrame is printed. This SFrame is lazily evaluated.
You can use sf.materialize() to force materialization.

In [251]:
# pubmed = load_sframe("Data/pubmed/pubmed.sframe")
# diseases_pubmed = fiter_diseases(pubmed,  disease_set, max_len)

# diseases_pubmed = diseases_pubmed.rename({"pubdate":"year"})
diseases_pubmed = diseases_pubmed.join(diseases_id, on={"disease":"diseases"})
diseases_pubmed.save("Data/pubmed/diseases_pubmed.sframe")

In [212]:
diseases_pubmed["Title"] = diseases_pubmed["Title"] 

abstract,affiliations,authors,chemical_list,country
,,LM Kireeva,,Ukraine
,,VG Goncharov,D001639:Bicarbonates;D000641:Ammonia ...,Ukraine
The emergence ofampicillin-resistant ...,,JD Nelson;CM Ginsburg;JCClahsen;LH Jackson ...,D000900:Anti-BacterialAgents; ...,United States
We have studied the roleof sodium ions in methyl ...,,GM van Thienen;PWPostma;K van Dam ...,D026901:MembraneTransport Proteins; ...,Netherlands
,,M Martínez-Báez,,Mexico
,,B Krankenhagen;MWessels;HJ Lehmann ...,D000900:Anti-BacterialAgents ...,Germany
This study characterizesthe response of BALB/c ...,,R Speirs;RW Benson;GSchiffman ...,D004168:DiphtheriaToxoid; D010567:Pertu ...,United States
This study characterizesthe response of BALB/c ...,,R Speirs;RW Benson;GSchiffman ...,D004168:DiphtheriaToxoid; D010567:Pertu ...,United States
This study characterizesthe response of BALB/c ...,,R Speirs;RW Benson;GSchiffman ...,D004168:DiphtheriaToxoid; D010567:Pertu ...,United States
Commercial neutral red(NR) originally ...,,GS Yen;EH Simon,D010619:Phenazines;D009499:Neutral Red ...,England


## Aminer

In [33]:
from ScienceDynamics.datasets.aminer import Aminer
from ScienceDynamics.config.configs import DATASETS_AMINER_DIR

In [34]:
aminer = Aminer(DATASETS_AMINER_DIR)
miner_sf = aminer.data

In [35]:
len(miner_sf)

172209563

In [215]:
miner_sf["Normalized paper title"] =  miner_sf["title"].apply(lambda x: x.lower())

In [217]:
miner_sf = miner_sf.rename({"id":"pub_id"})

In [218]:
diseases_miner = fiter_diseases(miner_sf,  disease_set, max_len)

In [219]:
diseases_miner = diseases_miner.join(diseases_id, on={"disease":"diseases"})

In [220]:
diseases_miner["year"] = diseases_miner["year"].astype(int)

In [253]:
diseases_miner.save("Data/aminer/diseases_aminer.sframe")

In [252]:
diseases_miner

abstract,authors,doi,pub_id,isbn
Using the total RNAextracted from ConA ...,"[{'name': 'TONG Tie-gang', 'org': 'National ...",None,53e9b732b7602d97042c9bd5,None
None,"[{'name': 'S. A. HERZOG','id': '53f432a7dabfae ...",10.1017/S0950268810001664,53e9b732b7602d97042c9c28,None
Objective To study thechange in Th1/Th2 ...,"[{'name': 'ZHU Bin','org': 'The infectious ...",None,53e9b732b7602d97042c9c33,None
None,"[{'name': 'Pamina M.Gorbach', 'id': '53f4 ...",10.1097/00002030-200216004-00006 ...,53e9b732b7602d97042c9c4b,None
We have previouslydemonstrated that ...,"[{'name': 'N Enomoto','id': '53f80937dabfae ...",10.1172/JCI118025,53e9b732b7602d97042c9c90,None
None,"[{'name': 'Naresh C.Bal', 'id': '53f42c8e ...",10.1515/BC.2007.057,53e9b732b7602d97042c9c91,None
To determine whetherrewording postal remi ...,"[{'name': 'P Hawe', 'id':'53f7b2e0dabfae938c6d ...",10.1136/adc.79.2.136,53e9b732b7602d97042c9cd4,None
"Microcin 140, a novelantibiotic of low ...","[{'name': 'Antonio F.Duro', 'org': ...",10.1016/0006-291X(79)91729-7 ...,53e9b732b7602d97042c9cf4,None
None,"[{'name': 'Raymond T.Chung', 'id': '53f453 ...",10.1002/hep.22070,53e9b732b7602d97042c9e35,None
None,"[{'name': 'HideoNakamura', 'org': ...",10.1016/S1072-7515(00)00589-5 ...,53e9b732b7602d97042c9ed7,None


## Semantic Scholar

In [185]:
!wget https://s3-us-west-2.amazonaws.com/ai2-s2-research-public/open-corpus/2019-11-01/manifest.txt


--2019-12-09 15:43:14--  https://s3-us-west-2.amazonaws.com/ai2-s2-research-public/open-corpus/2019-11-01/manifest.txt
Resolving s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)... 52.218.235.32
Connecting to s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)|52.218.235.32|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3075 (3.0K) [text/plain]
Saving to: 'manifest.txt'

100%[======================================>] 3,075       --.-K/s   in 0s      

2019-12-09 15:43:15 (56.5 MB/s) - 'manifest.txt' saved [3075/3075]



In [76]:
sf = SFrame.read_json("/data/semantic-scholar/sample-S2-records.gz", orient="lines")

TypeError: __init__() got an unexpected keyword argument 'orient'

In [77]:
from pathlib import Path

In [94]:
import time

In [99]:
for p in tqdm(Path("/data/semantic-scholar/").glob("s2-corpus-*.gz")):
    time.sleep(0.1)

In [ ]:
# %%capture output
for p in tqdm(Path("/data/semantic-scholar/").glob("s2-corpus-*.gz"),total=179):
    sf = SFrame.read_json(str(p), orient="lines")
    sf = sf[["id","sources","title","venue","year", "pmid","outCitations","journalName","inCitations","authors"]]
    sf.save(f"/data/semantic-scholar/temp/{p.stem}.sframe")

Finished parsing file /data/semantic-scholar/s2-corpus-004.gz

Parsing completed. Parsed 100 lines in 1.16358 secs.

------------------------------------------------------
Inferred types from first 100 line(s) of file as 
column_type_hints=[dict]
If parsing fails due to incorrect types, you can correct
the inferred type list above and pass it to read_csv in
the column_type_hints argument
------------------------------------------------------


Read 33508 lines. Lines per second: 16353.5

Read 199607 lines. Lines per second: 27400.3

...

Read 199840 lines. Lines per second: 26887.5

Read 366838 lines. Lines per second: 28578.1

Read 533181 lines. Lines per second: 29209.1

Read 700690 lines. Lines per second: 29814.4

Read 867633 lines. Lines per second: 30007.9

In [249]:
sf = SFrame.read_json("/data/semantic-scholar/s2-corpus-*.gz", orient="lines")

Finished parsing file /data/semantic-scholar/s2-corpus-004.gz

Parsing completed. Parsed 100 lines in 1.07844 secs.

------------------------------------------------------
Inferred types from first 100 line(s) of file as 
column_type_hints=[dict]
If parsing fails due to incorrect types, you can correct
the inferred type list above and pass it to read_csv in
the column_type_hints argument
------------------------------------------------------


Read 33508 lines. Lines per second: 17054.6

Read 199607 lines. Lines per second: 27485.1

...

Read 177991485 lines. Lines per second: 31079.4

Read 178157344 lines. Lines per second: 31080.9

Finished parsing file /data/semantic-scholar/s2-corpus-177.gz

Parsing completed. Parsed 178320056 lines in 5736.04 secs.

MemoryError: std::bad_alloc

In [109]:
full_sf =SFrame()
for p in tqdm(Path("/data/semantic-scholar/temp/").glob("s2-corpus-*.sframe"),total=179):
    full_sf = full_sf.append(load_sframe(str(p)))

In [111]:
full_sf.save("/data/semantic-scholar/s2-corpus.sframe")

In [32]:
sm = load_sframe("/data/semantic-scholar/s2-corpus.sframe")

In [33]:
sm = sm[sm["outCitations"]!=[]]
sm = sm.remove_column("outCitations")

In [ ]:
year_papaper = {row["id"]:row["year"] for row in tqdm(sm)}
year_papaper =  defaultdict(int, year_papaper)

In [ ]:
from turicreate import SArray
res =[]
for cite in tqdm(sm["outCitations"]):
    res.append({p:year_papaper[p] for p in cite})
sa = SArray(res)

In [ ]:
sm["inCitations"] = sa

In [ ]:
sm.save("/data/semantic-scholar/s2-corpus-small.sframe")

In [34]:
sm = load_sframe("/data/semantic-scholar/s2-corpus-small.sframe")

In [76]:
sm.head(1)

Columns:
	id	str
	sources	list
	title	str
	venue	str
	year	int
	journalName	str
	inCitations	dict
	authors	list

Rows: 1

Data:
+-------------------------------+-----------+-------------------------------+
|               id              |  sources  |             title             |
+-------------------------------+-----------+-------------------------------+
| 9d3f41517d56ac22fcc14cd637... | [Medline] | A novel serovar of Shigell... |
+-------------------------------+-----------+-------------------------------+
+-------------------------------+------+-------------------------------+
|             venue             | year |          journalName          |
+-------------------------------+------+-------------------------------+
| Journal of medical microbi... | 2007 | Journal of medical microbi... |
+-------------------------------+------+-------------------------------+
+-------------------------------+-------------------------------+
|          inCitations          |            authors            |
+-------------------------------+-------------------------------+
| {'941535d5797d7ad14ca1273d... | [{'name': 'Kaisar Ali Talu... |
+-------------------------------+-------------------------------+
[1 rows x 8 columns]

In [35]:
sm["Normalized paper title"] = sm["title"].apply(lambda x: x.lower().replace("[","").replace("].",""))

In [36]:
diseases_sm = fiter_diseases(sm, disease_set, max_len)

In [79]:
diseases_sm

id,sources,title,venue,year
c491f893ebdd0b423e9a5f4689382f535670dfff ...,[Medline],Longitudinal genomicsurveillance of ...,Genome Biology,2017
fc262595d39cc0c52dab974b062f51fd5f2c8ef3 ...,[Medline],Recurrence Prevention inPatients with Urinary ...,TheScientificWorldJournal,2004
0f1951b30eaf1bca60d50f0b44b291589c98ced5 ...,[],HIV/AIDS: The ZimbabweanSituation and Trends ...,,2013
0f1951b30eaf1bca60d50f0b44b291589c98ced5 ...,[],HIV/AIDS: The ZimbabweanSituation and Trends ...,,2013
3ce84555a29993bdd3d2cf84a8e67ebeeb835eae ...,[Medline],Cost-EffectivenessAnalysis of Prophylactic ...,PharmacoEconomics,2011
2f7b33f603e80fb82edd631338340dd14dadbab8 ...,[Medline],"Human papillomavirusprevalence, viral load ...",The Brazilian journal ofinfectious diseases : an ...,2002
e5feb386f02782767e8af9c986e3a777174ab57f ...,[],Einführung 2Inhaltsverzeichnis 3 ...,,2014
bd592779a2227846ee3b81a0b846955b4d909498 ...,[Medline],Factors Associated withImmunological Discord ...,Japanese journal ofinfectious diseases ...,2015
00e1f9a87e4043c7ce6626155b572b68efebf5dd ...,[],Seroprevalence of HIV inANC clinic attendees and ...,,2017
b246cc0c3fe8fdff761383afcf8da052d5db88a9 ...,[Medline],The effects of earlysyphilis on CD4 counts ...,Sexually transmittedinfections ...,2005


In [37]:
diseases_sm = diseases_sm.rename({"id":"paper id"})

In [38]:
diseases_sm = diseases_sm.join(diseases_id, on={"disease":"diseases"})

In [39]:
diseases_sm = diseases_sm.remove_column("disease")
diseases_sm = diseases_sm.join(gideon_diseases[["GIDEON","id"]], on={"id":"id"})
diseases_sm = diseases_sm.rename({"GIDEON":"disease"})

In [43]:
len(diseases_list)

334

In [44]:
diseases_sm

paper id,sources,title,venue,year
c491f893ebdd0b423e9a5f4689382f535670dfff ...,[Medline],Longitudinal genomicsurveillance of ...,Genome Biology,2017
fc262595d39cc0c52dab974b062f51fd5f2c8ef3 ...,[Medline],Recurrence Prevention inPatients with Urinary ...,TheScientificWorldJournal,2004
0f1951b30eaf1bca60d50f0b44b291589c98ced5 ...,[],HIV/AIDS: The ZimbabweanSituation and Trends ...,,2013
0f1951b30eaf1bca60d50f0b44b291589c98ced5 ...,[],HIV/AIDS: The ZimbabweanSituation and Trends ...,,2013
3ce84555a29993bdd3d2cf84a8e67ebeeb835eae ...,[Medline],Cost-EffectivenessAnalysis of Prophylactic ...,PharmacoEconomics,2011
2f7b33f603e80fb82edd631338340dd14dadbab8 ...,[Medline],"Human papillomavirusprevalence, viral load ...",The Brazilian journal ofinfectious diseases : an ...,2002
e5feb386f02782767e8af9c986e3a777174ab57f ...,[],Einführung 2Inhaltsverzeichnis 3 ...,,2014
bd592779a2227846ee3b81a0b846955b4d909498 ...,[Medline],Factors Associated withImmunological Discord ...,Japanese journal ofinfectious diseases ...,2015
00e1f9a87e4043c7ce6626155b572b68efebf5dd ...,[],Seroprevalence of HIV inANC clinic attendees and ...,,2017
b246cc0c3fe8fdff761383afcf8da052d5db88a9 ...,[Medline],The effects of earlysyphilis on CD4 counts ...,Sexually transmittedinfections ...,2005


In [47]:
pmid = load_sframe("/data/semantic-scholar/s2-pid.sframe")

In [51]:
diseases_sm = diseases_sm.join(pmid, {"paper id":"id"})

In [52]:
diseases_sm

paper id,sources,title,venue,year
c491f893ebdd0b423e9a5f4689382f535670dfff ...,[Medline],Longitudinal genomicsurveillance of ...,Genome Biology,2017
fc262595d39cc0c52dab974b062f51fd5f2c8ef3 ...,[Medline],Recurrence Prevention inPatients with Urinary ...,TheScientificWorldJournal,2004
0f1951b30eaf1bca60d50f0b44b291589c98ced5 ...,[],HIV/AIDS: The ZimbabweanSituation and Trends ...,,2013
0f1951b30eaf1bca60d50f0b44b291589c98ced5 ...,[],HIV/AIDS: The ZimbabweanSituation and Trends ...,,2013
3ce84555a29993bdd3d2cf84a8e67ebeeb835eae ...,[Medline],Cost-EffectivenessAnalysis of Prophylactic ...,PharmacoEconomics,2011
2f7b33f603e80fb82edd631338340dd14dadbab8 ...,[Medline],"Human papillomavirusprevalence, viral load ...",The Brazilian journal ofinfectious diseases : an ...,2002
e5feb386f02782767e8af9c986e3a777174ab57f ...,[],Einführung 2Inhaltsverzeichnis 3 ...,,2014
bd592779a2227846ee3b81a0b846955b4d909498 ...,[Medline],Factors Associated withImmunological Discord ...,Japanese journal ofinfectious diseases ...,2015
00e1f9a87e4043c7ce6626155b572b68efebf5dd ...,[],Seroprevalence of HIV inANC clinic attendees and ...,,2017
b246cc0c3fe8fdff761383afcf8da052d5db88a9 ...,[Medline],The effects of earlysyphilis on CD4 counts ...,Sexually transmittedinfections ...,2005


In [53]:
diseases_sm.save("Data/sm/diseases_sm.sframe")